# Chapter 2: Pandas — Core Skills

### Why This Matters

If NumPy is the engine, **Pandas is the dashboard, steering wheel, and seats** — it's the library you'll actually spend most of your working hours in as an analyst. Pandas gives you the `DataFrame`: a table with named, differently-typed columns and flexible row labels, built specifically to handle the messy reality of business data (missing values, mixed types, dates, categories) that NumPy alone struggles with. If you only master one library from this entire course to the point of fluency, make it this one.

### 2.1 Creating and Exploring DataFrames

In [ ]:
import pandas as pd
import numpy as np

data = {
    "Region": ["North", "South", "East", "West", "North", "South"],
    "Month":  ["Jan", "Jan", "Jan", "Jan", "Feb", "Feb"],
    "Sales":  [1250000, 980000, 1120000, 1450000, 1320000, 1050000],
    "Target": [1200000, 1000000, 1100000, 1400000, 1250000, 1000000]
}
df = pd.DataFrame(data)
df

A **DataFrame** is best thought of as a dictionary of columns, where each column is technically a **Series** (a 1D labeled array — essentially a NumPy array with an index attached). Building one from a dictionary, as above, is the most common way to construct small DataFrames by hand; in real work you'll far more often *load* one from a file (Section 2.6).

In [ ]:
print(df.shape)              # (6, 4) -> (rows, columns)
print(df.columns.tolist())   # ['Region', 'Month', 'Sales', 'Target']
print(df.dtypes)             # the data type Pandas inferred for each column

df.head()        # first 5 rows (or df.head(10) for a specific count)
df.info()        # column names, non-null counts, dtypes, memory usage — always run this on new data
df.describe()    # count/mean/std/min/quartiles/max for every numeric column

💡 **These four commands — `.head()`, `.info()`, `.describe()`, `.shape` — should be the *automatic, unthinking first move* on any dataset you're handed**, before you write a single line of actual analysis. `.info()` in particular tells you immediately whether a column you expect to be numeric actually loaded as text (a classic sign of stray characters like currency symbols or commas in the source file), and whether any columns have missing values, both of which will bite you downstream if unnoticed.

### 2.2 Selecting and Filtering Data

In [ ]:
df["Sales"]                    # a single column -> returns a Series
df[["Region", "Sales"]]        # multiple columns -> note the DOUBLE brackets, returns a DataFrame

df[df["Sales"] > 1200000]                            # boolean filtering — identical logic to NumPy, Chapter 1
df[(df["Sales"] > 1100000) & (df["Region"] == "West")]   # combine conditions with & / | and parentheses

⚠️ The exact same rule from NumPy applies here, and it's worth repeating because it's the single most common Pandas error beginners hit: use `&` and `|`, never `and`/`or`, and wrap each condition in parentheses. `df[df["Sales"] > 1100000 & df["Region"] == "West"]` (missing parens) will raise a confusing error due to operator precedence — always write `(condition1) & (condition2)`.

In [ ]:
df.loc[0:2, ["Region", "Sales"]]     # LABEL-based selection (by index/column NAME)
df.iloc[0:3, 0:2]                    # POSITION-based selection (by integer position, like a NumPy array)

**`.loc` vs `.iloc` is a distinction every beginner mixes up at least once**, so it's worth being explicit: `.loc` selects by the *label* (the name of the row/column) — and, notably, when using `.loc` with a slice, the endpoint is **inclusive** (`0:2` gives you rows labeled 0, 1, *and* 2). `.iloc` selects by *integer position*, exactly like NumPy array indexing, where the endpoint is **exclusive** (`0:3` gives you positions 0, 1, 2 — three rows). If your DataFrame's index is the default `0, 1, 2, ...`, `.loc` and `.iloc` often look deceptively similar for row selection, which is precisely why they're confusing — the difference becomes obvious and important the moment your index is something else (e.g., dates, or customer IDs, or after you've filtered/sorted the DataFrame and the row labels no longer match their positions).

### 2.3 Creating New Columns (Feature Engineering)

In [ ]:
df["Achievement"] = (df["Sales"] / df["Target"]) * 100
df["Variance"] = df["Sales"] - df["Target"]
df["Status"] = np.where(df["Achievement"] >= 100, "Met", "Missed")
df

`np.where(condition, value_if_true, value_if_false)` is the vectorized equivalent of a ternary expression, applied to an entire column at once — this is, functionally, the Pandas/NumPy answer to writing an `if/else` for every row without an actual loop. You'll use this constantly to create categorical flags from numeric thresholds ("High/Medium/Low," "Met/Missed," "Churned/Active").

**This is "feature engineering"** — deriving new, more useful columns from existing ones — and it's arguably the single highest-leverage skill in applied data analysis and machine learning. A raw `Sales` and `Target` column are individually informative; `Achievement` (their ratio) is often *more* informative because it normalizes for scale and directly answers the business question ("did we hit target?").

### 2.4 Sorting

In [ ]:
df.sort_values("Sales", ascending=False)
df.sort_values(["Region", "Sales"], ascending=[True, False])   # sort by Region first, then Sales within each Region

### 2.5 Aggregation and GroupBy — The Heart of Business Analysis

In [ ]:
df.groupby("Region")["Sales"].sum()      # total sales, one number per region

df.groupby("Region").agg({
    "Sales": ["sum", "mean", "max"],
    "Target": "sum",
    "Achievement": "mean"
}).round(1)

pd.pivot_table(df, values="Sales", index="Region", columns="Month", aggfunc="sum", fill_value=0)

**`groupby` follows a "split-apply-combine" pattern** that's worth understanding conceptually, not just as syntax: Pandas *splits* the DataFrame into separate groups (one per unique Region value), *applies* your aggregation function (`.sum()`, `.mean()`, etc.) independently within each group, then *combines* the per-group results back into a single output. This three-step mental model — split, apply, combine — is exactly how Excel's PivotTables work too, which is precisely why the source material calls Pandas "Excel-like power with Python's speed": if you've ever built a PivotTable, `groupby` + `agg` is that same operation, just expressed as code instead of clicks, and therefore fully automatable and reproducible.

`.agg({...})` lets you specify *different* aggregation functions for different columns in one call — genuinely central to any real reporting task, where you typically want a *sum* of revenue but an *average* of a rate or percentage in the same summary table.

`pd.pivot_table` is Pandas' direct equivalent of Excel's PivotTable feature: `index` becomes your rows, `columns` becomes your... columns, `values` is what gets aggregated, and `aggfunc` is how. `fill_value=0` replaces any resulting missing combinations (e.g., a region with no sales in a given month) with 0 instead of `NaN`.

### 2.6 Reading Real Files

In [ ]:
# The single most common line of code you will write for the rest of your career:
df = pd.read_csv("sales_data.csv")
df = pd.read_excel("sales_data.xlsx", sheet_name="Sheet1")

# Saving back out
df.to_csv("temp_sales.csv", index=False)     # index=False avoids writing the DataFrame's row numbers as a column
df_reloaded = pd.read_csv("temp_sales.csv")

💡 `index=False` on `.to_csv()` is worth internalizing as a near-default habit — without it, Pandas writes the DataFrame's row index (0, 1, 2, ...) as an extra unnamed column in the file, which is almost never what you want when handing the CSV to someone else or re-reading it later.

`pd.read_csv` and `pd.read_excel` both accept a long list of optional keyword arguments for real-world messiness: `sep=";"` for non-comma delimiters, `encoding="latin-1"` for files that aren't UTF-8, `parse_dates=["Date"]` to automatically convert a column to proper datetime objects on load, `na_values=["N/A", "-"]` to tell Pandas which text strings in the raw file should be treated as missing data rather than literal text. When a file doesn't load cleanly on the first try, these arguments — not a manual cleanup pass afterward — are usually the right fix.

### 2.7 Handling Missing Data

In [ ]:
df2 = df.copy()          # .copy() creates an INDEPENDENT copy — editing df2 will NOT affect the original df
df2.loc[1, "Sales"] = np.nan
df2.loc[3, "Target"] = np.nan

print(df2.isna().sum())              # count of missing values PER COLUMN — always check this before analyzing

df2.dropna()                                          # drop any row containing at least one NaN
df2["Sales"].fillna(df2["Sales"].mean(), inplace=False)   # fill with the column mean instead of dropping

⚠️ **`.copy()` matters more than it looks.** `df2 = df` (without `.copy()`) doesn't create a new DataFrame — it creates a second name pointing at the *exact same* underlying data, so modifying `df2` would silently also modify `df`. This is a direct consequence of Python's variable-as-a-label model from Volume 01, and it is a genuinely common source of "why did my original data change" bugs. Whenever you intend to experiment on a variant of a DataFrame while preserving the original, use `.copy()` explicitly.

⚠️ **`inplace=True` vs. reassignment** is a related, equally common trap. `df2["Sales"].fillna(value, inplace=False)` (the default) *returns a new Series* and leaves `df2` unchanged unless you capture it: `df2["Sales"] = df2["Sales"].fillna(value)`. Many Pandas methods offer an `inplace=True` option that mutates the DataFrame directly and returns `None` instead — but modern Pandas guidance (and most style guides) now lean *against* `inplace=True` in general, because chaining and error-tracing become harder when methods return nothing. The safest universal habit: always reassign (`df = df.method(...)`) rather than relying on `inplace=True`.

**Choosing how to handle missing data is itself an analytical decision, not just a technical one** — dropping rows is safe but throws away potentially valuable data (and can bias your dataset if missingness isn't random); filling with the mean is common for numeric columns but can understate real variability; filling with a specific business-meaningful default (e.g., 0 for "no purchases" or "Unknown" for a missing category) is often more honest than either. There's no universal right answer — the right choice depends on *why* the data is missing, which is a business/domain question, not a syntax question.

### 2.8 Merging DataFrames (Joins)

In [ ]:
region_info = pd.DataFrame({
    "Region": ["North", "South", "East", "West"],
    "Manager": ["Anita", "Ravi", "Priya", "Vikram"],
    "Zone": ["A", "B", "A", "B"]
})

merged = pd.merge(df, region_info, on="Region", how="left")

`pd.merge` is Pandas' equivalent of a SQL `JOIN` — and if you already know SQL joins (which, given your background touching databases, you likely do to some degree), the vocabulary transfers directly: `on="Region"` is the join key (equivalent to a SQL `ON` clause), and `how` controls join type:

| `how=` | SQL equivalent | Keeps |
|---|---|---|
| `"left"` | LEFT JOIN | Every row from the left DataFrame, matching data from the right where it exists (else `NaN`) |
| `"right"` | RIGHT JOIN | Every row from the right DataFrame |
| `"inner"` | INNER JOIN | Only rows where the key exists in **both** DataFrames |
| `"outer"` | FULL OUTER JOIN | Every row from **both**, with `NaN` wherever a match is missing |

`how="left"` is the most common default in reporting workflows: "keep every one of my transactions, and attach manager/zone info where I have it."

### Cheat Sheet — Pandas Core

| Task | Code |
|---|---|
| Load | `pd.read_csv(path)`, `pd.read_excel(path, sheet_name=...)` |
| Explore | `.head() .info() .describe() .shape .dtypes` |
| Select column(s) | `df["col"]`, `df[["col1","col2"]]` |
| Filter rows | `df[(cond1) & (cond2)]` |
| Label vs position | `.loc[label]` (inclusive slice), `.iloc[pos]` (exclusive slice) |
| New column | `df["new"] = expr`, `np.where(cond, a, b)` |
| Sort | `.sort_values("col", ascending=False)` |
| Group + aggregate | `.groupby("col")["x"].sum()`, `.groupby("col").agg({...})` |
| Pivot | `pd.pivot_table(df, values, index, columns, aggfunc)` |
| Missing data | `.isna().sum()`, `.dropna()`, `.fillna(value)` |
| Copy safely | `df2 = df.copy()` |
| Join | `pd.merge(left, right, on="key", how="left")` |

### 🎯 Business Challenge — Pandas Core

*Task:* Build a 12+ row sales dataset (`Date, Region, Product, Units, Unit_Price`); compute revenue; totals by region/product; top product; average price by region; a High/Normal status column; export to CSV.

In [ ]:
import pandas as pd
import numpy as np

data = {
    "Date": pd.date_range("2025-01-01", periods=12, freq="W"),
    "Region": ["North","South","East","West"]*3,
    "Product": ["Laptop","Mouse","Keyboard","Monitor"]*3,
    "Units": [12, 45, 30, 8, 15, 50, 28, 10, 18, 40, 35, 9],
    "Unit_Price": [62500, 799, 1450, 11200, 62500, 799, 1450, 11200, 62500, 799, 1450, 11200]
}
df = pd.DataFrame(data)

df["Revenue"] = df["Units"] * df["Unit_Price"]

revenue_by_region = df.groupby("Region")["Revenue"].sum().sort_values(ascending=False)
revenue_by_product = df.groupby("Product")["Revenue"].sum().sort_values(ascending=False)
top_product = revenue_by_product.idxmax()      # idxmax -> the LABEL of the max value, not the value itself

avg_price_by_region = df.groupby("Region")["Unit_Price"].mean()
df["Status"] = np.where(df["Revenue"] > 500000, "High", "Normal")

print("Revenue by region:\n", revenue_by_region)
print("\nTop product by revenue:", top_product)
print("\nAverage price by region:\n", avg_price_by_region.round(2))

df.to_csv("sales_challenge_output.csv", index=False)
print("\nSaved to sales_challenge_output.csv")

💡 `.idxmax()` (index of the maximum) vs. `.max()` (the maximum value itself) is a distinction worth remembering — you'll want `idxmax` any time the *question* is "which one" rather than "what's the highest number."

---